# Comparing the N-body implementations

Each implementation runs in its own folder and saves its settings, timings and
animation there, while this notebook only reads those outputs. Run the main
simulation, animation and scaling cells in each variant notebook first; no source
is compiled and no simulation is started here.

All five use the ring and settings in [comparison_config.json](./comparison_config.json).
The main run uses 10,000 bodies and 1,000 steps with a frame every two steps, while
the scaling sweep uses 100 steps at each shared body count without saving frames.
Changing the configuration means rerunning the variant notebooks before comparing
their outputs. Missing exports or exports with different settings are listed rather
than treated as measurements.

In [ ]:
from pathlib import Path
import os
import sys
import base64
import html
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

# Change this if your Drive clone is somewhere else.
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/projects/n-body-simulation")
try:
    from google.colab import drive
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()
    if not (PROJECT_ROOT / "comparison_config.json").is_file():
        PROJECT_ROOT = PROJECT_ROOT.parent
else:
    drive.mount("/content/drive")
    PROJECT_ROOT = DRIVE_PROJECT_ROOT
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("Project folder:", PROJECT_ROOT)

from benchmark import VARIANTS, load_export, load_settings

SETTINGS = load_settings(PROJECT_ROOT)
COLORS = {
    "CPU": "#37424c", "Naive CUDA": "#3d709c", "Tiled CUDA": "#4c8273",
    "Optimised CUDA": "#986f40", "Barnes-Hut CUDA": "#806b9c",
}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
display(pd.DataFrame([
    {"run": "Animation", **SETTINGS["main"]},
    {"run": "Benchmark", **SETTINGS["benchmark"]},
]))
display(pd.Series({
    "seed": SETTINGS["seed"], "time step": SETTINGS["time_step"],
    "G": SETTINGS["gravitational_constant"], "squared softening": SETTINGS["softening"],
}, name="Shared physical inputs").to_frame())

## Available outputs

Run the variant notebooks before comparing, and rerun them after changing settings
or source. This notebook reads the saved CSVs and metadata directly; it does not
validate every column or recheck file checksums.

In [ ]:
availability = []
benchmark_metadata = {}
main_exports = {}
benchmark_tables = []
for variant, specification in VARIANTS.items():
    status = {"variant": specification["label"]}
    for kind in ["main", "benchmark"]:
        folder = PROJECT_ROOT / specification["folder"]
        path = folder / ("run.json" if kind == "main" else "scaling.json")
        if not path.is_file():
            status[kind] = "Run the variant notebook"
        else:
            metadata = load_export(PROJECT_ROOT, variant, kind)
            if metadata["configuration"] != SETTINGS:
                status[kind] = "Different settings; rerun the variant notebook"
            elif kind == "main":
                main_exports[variant] = metadata
                status[kind] = "Ready"
                status["animation"] = "Ready" if "nbody.gif" in metadata["artifacts"] else "Render the GIF"
            elif variant in main_exports and (
                metadata["executable_sha256"] != main_exports[variant]["executable_sha256"]
                or metadata["initial_state_id"] != main_exports[variant]["initial_state_id"]
            ):
                status[kind] = "Main and benchmark exports come from different builds or starting states; rerun scaling."
            else:
                table = pd.read_csv(folder / "scaling.csv").sort_values("bodies")
                benchmark_tables.append(table)
                benchmark_metadata[variant] = metadata
                status[kind] = "Ready"
        if status[kind] != "Ready":
            warnings.warn(f"{specification['label']} {kind}: {status[kind]}", stacklevel=1)
    availability.append(status)
display(pd.DataFrame(availability).set_index("variant").fillna("Unavailable"))
results = pd.concat(benchmark_tables, ignore_index=True) if benchmark_tables else pd.DataFrame()
if results.empty:
    display(Markdown("**No compatible benchmark outputs yet.** Run the variant notebooks, then rerun this notebook."))

## Hardware and timing

The compute columns are useful for following how each implementation scales, but
their timers do not cover exactly the same work. CPU uses host wall time, naive
and tiled use CUDA kernel events, optimised uses events around its queued kernels,
and Barnes-Hut includes sorting, tree building, force calculation and phase waits.
The common benchmark disables frame saving, so buffered frame copies are not part
of the optimised measurement.

Total time is the executable's own wall time, including setup, tuning where used
and final output. Compilation, GIF rendering and notebook overhead are outside it.
Speedups are only shown when the exported hardware and starting-state checks match,
since different machines or starting bodies would change more than the implementation.

In [ ]:
hardware_rows = []
for variant, metadata in benchmark_metadata.items():
    hardware_rows.append({
        "variant": VARIANTS[variant]["label"], **metadata["hardware"],
        "recorded": metadata["created_at"], "compute timer": metadata["timing_scope"],
    })
if hardware_rows:
    display(pd.DataFrame(hardware_rows).set_index("variant"))

same_hardware = len({
    json.dumps(metadata["hardware"], sort_keys=True) for metadata in benchmark_metadata.values()
}) == 1
same_initial_state = len({metadata["initial_state_id"] for metadata in benchmark_metadata.values()}) == 1
speedups_allowed = bool(benchmark_metadata) and same_hardware and same_initial_state
if benchmark_metadata and not speedups_allowed:
    warnings.warn("Hardware or starting bodies differ between exports. Timing charts remain available, but speedups are omitted.")

## Time per step and total runtime

Both charts use measured values at the same body counts, with no projected points.
The quadratic reference follows the largest measured CPU point when it is available,
and shows growth rather than predicting another implementation's runtime.

In [ ]:
if not results.empty:
    order = [specification["label"] for specification in VARIANTS.values()
             if specification["label"] in results["version"].unique()]
    per_step = results.pivot(index="bodies", columns="version", values="ms per step").reindex(columns=order)
    total_time = results.pivot(index="bodies", columns="version", values="total ms").reindex(columns=order)
    display(Markdown("**Compute time per step (ms)**"))
    display(per_step.round(6))
    display(Markdown("**Total executable time (ms)**"))
    display(total_time.round(3))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
    for axis, table, title in [
        (axes[0], per_step, "Compute time per step"), (axes[1], total_time, "Total executable time")
    ]:
        for label in order:
            axis.plot(table.index, table[label], "o-", color=COLORS[label], label=label)
        axis.set_xscale("log", base=2)
        axis.set_yscale("log")
        axis.set_xlabel("Number of bodies")
        axis.set_ylabel("Time (ms)")
        axis.set_title(title)
        axis.grid(True, which="both", alpha=0.2)
    if "CPU" in per_step:
        reference = per_step["CPU"].iloc[-1] * (per_step.index.to_numpy() / per_step.index[-1]) ** 2
        axes[0].plot(per_step.index, reference, "--", color="#87929c", label="N squared reference")
    axes[0].legend(fontsize=8)
    plt.show()

## Speedup over the CPU

A value above one means the GPU version took less time than the CPU at that measured
body count. Compute and total-runtime ratios are shown separately because setup and
tuning can change which version finishes first, especially for small runs. The
first faster point is only a measured sample, not an exact crossover threshold.

In [ ]:
if not results.empty and "CPU" in per_step and len(per_step.columns) > 1 and speedups_allowed:
    compute_speedup = per_step.rdiv(per_step["CPU"], axis=0).drop(columns="CPU")
    total_speedup = total_time.rdiv(total_time["CPU"], axis=0).drop(columns="CPU")
    display(Markdown("**Compute speedup**"))
    display(compute_speedup.round(3))
    display(Markdown("**Total-runtime speedup**"))
    display(total_speedup.round(3))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
    crossovers = []
    for label in compute_speedup:
        for axis, table in [(axes[0], compute_speedup), (axes[1], total_speedup)]:
            axis.plot(table.index, table[label], "o-", color=COLORS[label], label=label)
        compute_faster = compute_speedup.index[compute_speedup[label] > 1]
        total_faster = total_speedup.index[total_speedup[label] > 1]
        crossovers.append({
            "variant": label,
            "first faster compute sample": int(compute_faster[0]) if len(compute_faster) else "Not observed",
            "first faster total-time sample": int(total_faster[0]) if len(total_faster) else "Not observed",
        })
    for axis, title in zip(axes, ["Compute speedup", "Total-runtime speedup"]):
        axis.axhline(1, color="#87929c", linestyle="--", label="CPU baseline")
        axis.set_xscale("log", base=2)
        axis.set_xlabel("Number of bodies")
        axis.set_ylabel("CPU time / GPU time")
        axis.set_title(title)
        axis.grid(True, alpha=0.2)
        axis.legend(fontsize=8)
    plt.show()
    display(pd.DataFrame(crossovers).set_index("variant"))
elif not results.empty:
    display(Markdown("Speedups need a compatible CPU export and matching hardware and starting bodies."))

## Animations from each folder

These are the saved GIFs rather than simulations rerun by this notebook. Every
variant uses the same x-y projection, frame steps, mass coloring and playback rate;
the view follows the main group, so distant bodies can leave the frame. Each GIF
still chooses its own view width as its system spreads, so compare the axis scales
as well as the visible shape.

In [ ]:
cards = []
for variant, specification in VARIANTS.items():
    metadata = main_exports.get(variant)
    if metadata is None or "nbody.gif" not in metadata["artifacts"]:
        continue
    path = PROJECT_ROOT / specification["folder"] / "nbody.gif"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    label = html.escape(specification["label"])
    cards.append(
        f'<figure style="margin:0;flex:1 1 300px;max-width:480px">'
        f'<figcaption><strong>{label}</strong></figcaption>'
        f'<img alt="{label} saved N-body animation" style="width:100%;height:auto" '
        f'src="data:image/gif;base64,{encoded}"></figure>'
    )
if cards:
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:20px">' + "".join(cards) + "</div>"))
else:
    display(Markdown("No saved animations yet. Run the animation cells in the variant notebooks."))

## Final states

Body IDs put the saved rows back into the same order before positions are compared.
Floating-point arithmetic and summation order can separate direct trajectories over
many steps, while Barnes-Hut also approximates the force, so these differences are
descriptive rather than a pass/fail accuracy test. A short force check is more useful
for assessing the tree's opening angle than matching long trajectories point by point.

In [ ]:
final_states = {}
for variant, metadata in main_exports.items():
    folder = PROJECT_ROOT / VARIANTS[variant]["folder"]
    filename = "final_output.csv" if variant == "barnes_hut" else "final_state.csv"
    state = pd.read_csv(folder / filename).rename(columns={"i": "id", "m": "mass"})
    final_states[variant] = state.sort_values("id").reset_index(drop=True)

if final_states:
    columns = min(3, len(final_states))
    rows = (len(final_states) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows), squeeze=False,
        constrained_layout=True, sharex=True, sharey=True)
    radius = max(np.percentile(np.hypot(state["x"], state["y"]), 90) for state in final_states.values())
    radius = max(0.1, radius * 1.3)
    for axis, (variant, state) in zip(axes.flat, final_states.items()):
        label = VARIANTS[variant]["label"]
        axis.scatter(state["x"], state["y"], s=2, color=COLORS[label], alpha=0.5, linewidths=0)
        axis.set(xlim=(-radius, radius), ylim=(-radius, radius), xlabel="x", ylabel="y", title=label)
        axis.set_aspect("equal")
    for axis in list(axes.flat)[len(final_states):]:
        axis.set_visible(False)
    plt.show()

if "cpu" in final_states:
    reference = final_states["cpu"][["x", "y", "z"]].to_numpy()
    differences = []
    for variant, state in final_states.items():
        if variant == "cpu":
            continue
        if main_exports[variant]["initial_state_id"] != main_exports["cpu"]["initial_state_id"]:
            warnings.warn(f"{VARIANTS[variant]['label']}: starting bodies differ; final-position differences omitted.")
            continue
        distance = np.linalg.norm(state[["x", "y", "z"]].to_numpy() - reference, axis=1)
        differences.append({
            "variant": VARIANTS[variant]["label"],
            "RMS position difference": np.sqrt(np.mean(distance ** 2)),
            "median position difference": np.median(distance),
            "maximum position difference": distance.max(),
        })
    if differences:
        display(pd.DataFrame(differences).set_index("variant"))